In [1]:
'''Reconcile-HAR Basecode, 2026/03/26'''

import os, math, time, random
import numpy as np
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────────────────────
# Seed & Data
# ──────────────────────────────────────────────────────────────────────────────
def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_GPU = DEVICE.type == "cuda"
print(f"Device: {DEVICE} | pin_memory: {USE_GPU}")

SIGNALS = ["body_acc_x", "body_acc_y", "body_acc_z",
           "body_gyro_x", "body_gyro_y", "body_gyro_z",
           "total_acc_x", "total_acc_y", "total_acc_z"]

def load_uci_har_dataset(data_dir):
    data = {}
    for split in ['train', 'test']:
        inertial_dir = os.path.join(data_dir, split, "Inertial Signals")
        signals = [np.loadtxt(os.path.join(inertial_dir, f"{sig}_{split}.txt"), dtype=np.float32) for sig in SIGNALS]
        data[split] = {
            'X': np.stack(signals, axis=-1),
            'y': np.loadtxt(os.path.join(data_dir, split, f"y_{split}.txt"), dtype=np.int64) - 1
        }

    X_train_raw, y_train = data['train']['X'], data['train']['y']
    X_test_raw, y_test = data['test']['X'], data['test']['y']

    N_tr, T, C = X_train_raw.shape
    N_te = X_test_raw.shape[0]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw.reshape(-1, C)).reshape(N_tr, T, C)
    X_test = scaler.transform(X_test_raw.reshape(-1, C)).reshape(N_te, T, C)

    X_train = X_train.transpose(0, 2, 1)
    X_test = X_test.transpose(0, 2, 1)

    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    test_dataset = TensorDataset(
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long)
    )

    print(f"Train dataset size: {len(train_dataset)}, Test dataset size: {len(test_dataset)}")
    return train_dataset, test_dataset


# ──────────────────────────────────────────────────────────────────────────────
# Frequency Branch
# ──────────────────────────────────────────────────────────────────────────────
class MultiScaleLearnableSTFT(nn.Module):
    def __init__(self, in_channels, bins_per_scale=16, kernel_sizes=[32, 64], hop=8):
        super().__init__()
        self.banks = nn.ModuleList()
        self.norms = nn.ModuleList()
        for kernel_size in kernel_sizes:
            self.banks.append(
                nn.Conv1d(in_channels, bins_per_scale, kernel_size=kernel_size,
                          stride=hop, padding=kernel_size // 2, bias=False)
            )
            self.norms.append(nn.BatchNorm1d(bins_per_scale))

    def forward(self, x):
        specs = []
        for conv, bn in zip(self.banks, self.norms):
            c = conv(x)
            mag = torch.sqrt(c ** 2 + 1e-6)
            log_mag = bn(torch.log1p(mag))
            specs.append(log_mag)

        min_t = min(s.size(2) for s in specs)
        specs = [s[:, :, :min_t] for s in specs]
        return torch.cat(specs, dim=1).unsqueeze(1)


class FreqEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.GELU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Linear(32, out_dim), nn.LayerNorm(out_dim), nn.GELU()
        )

    def forward(self, x):
        h = self.net(x).view(x.size(0), -1)
        return self.fc(h)


# ──────────────────────────────────────────────────────────────────────────────
# Time Branch
# ──────────────────────────────────────────────────────────────────────────────
class PerceiverCrossAttention(nn.Module):
    def __init__(self, d_latent, d_input, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_latent, n_heads, batch_first=True, dropout=dropout)
        self.norm_q = nn.LayerNorm(d_latent)
        self.norm_kv = nn.LayerNorm(d_input)
        self.proj_kv = nn.Linear(d_input, d_latent) if d_input != d_latent else nn.Identity()

    def forward(self, latents, inputs):
        q = self.norm_q(latents)
        kv = self.proj_kv(self.norm_kv(inputs))
        out, _ = self.attn(q, kv, kv)
        return latents + out


class PerceiverSelfBlock(nn.Module):
    def __init__(self, d_latent, n_heads=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_latent)
        self.attn  = nn.MultiheadAttention(d_latent, n_heads, batch_first=True, dropout=dropout)
        self.norm2 = nn.LayerNorm(d_latent)
        self.ffn   = nn.Sequential(
            nn.Linear(d_latent, d_latent * ff_mult), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_latent * ff_mult, d_latent), nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.norm1(x)
        x = x + self.attn(h, h, h)[0]
        x = x + self.ffn(self.norm2(x))
        return x


class PerceiverStyleEncoder(nn.Module):
    def __init__(self, input_dim, latent_n=32, latent_d=128, n_heads=4,
                 num_blocks=2, output_dim=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, latent_d), nn.LayerNorm(latent_d)
        )
        self.pos_enc = nn.Parameter(torch.randn(1, 512, latent_d) * 0.02)
        self.latents = nn.Parameter(torch.randn(1, latent_n, latent_d) * 0.02)
        self.cross_attn = PerceiverCrossAttention(latent_d, latent_d, n_heads, dropout)
        self.self_attn_blocks = nn.ModuleList([
            PerceiverSelfBlock(latent_d, n_heads, ff_mult=2, dropout=dropout)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(latent_d)
        self.fc_out = nn.Sequential(
            nn.Linear(latent_d, output_dim), nn.LayerNorm(output_dim), nn.GELU()
        )

    def forward(self, x):
        B, T, _ = x.shape
        inp = self.input_proj(x) + self.pos_enc[:, :T, :]
        latents = self.latents.expand(B, -1, -1)
        latents = self.cross_attn(latents, inp)
        for block in self.self_attn_blocks:
            latents = block(latents)
        return self.fc_out(self.norm(latents).mean(dim=1))


# ──────────────────────────────────────────────────────────────────────────────
# Conflict Estimator & Fusion
# ──────────────────────────────────────────────────────────────────────────────
class ConflictEstimator(nn.Module):
    def __init__(self, n_features=5, lambda_floor=0.1, n_classes=6):
        super().__init__()
        self.lambda_floor = lambda_floor
        self.log_k = math.log(n_classes)
        self.mlp = nn.Sequential(
            nn.Linear(n_features, 16), nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, z_t, z_f, p_t, p_f):
        z_t_d = z_t.detach()
        z_f_d = z_f.detach()
        p_t_d = p_t.detach()
        p_f_d = p_f.detach()

        cos_sim = F.cosine_similarity(z_t_d, z_f_d, dim=-1, eps=1e-6)
        cos_norm = (cos_sim + 1.0) / 2.0

        l2_dist = torch.norm(z_t_d - z_f_d, dim=-1, p=2)
        denom = z_t_d.norm(dim=-1) + z_f_d.norm(dim=-1) + 1e-6
        l2_norm = (l2_dist / denom).clamp(0.0, 1.0)

        ent_t = (-(p_t_d * (p_t_d + 1e-8).log()).sum(dim=-1) / self.log_k).clamp(0.0, 1.0)
        ent_f = (-(p_f_d * (p_f_d + 1e-8).log()).sum(dim=-1) / self.log_k).clamp(0.0, 1.0)

        top1_agree = (p_t_d.argmax(dim=-1) == p_f_d.argmax(dim=-1)).float()

        features = torch.stack([cos_norm, l2_norm, ent_t, ent_f, top1_agree], dim=-1)
        raw = torch.sigmoid(self.mlp(features).squeeze(-1))
        lam = self.lambda_floor + (1.0 - self.lambda_floor) * raw
        return lam


class ConflictAwareFusion(nn.Module):
    def __init__(self, feat_dim, dropout=0.1):
        super().__init__()
        self.proj_t = nn.Linear(feat_dim, feat_dim)
        self.proj_f = nn.Linear(feat_dim, feat_dim)
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * 2 + 1, feat_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, feat_dim),
            nn.Sigmoid()
        )
        self.out_proj = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.LayerNorm(feat_dim),
            nn.GELU()
        )

    def forward(self, z_t, z_f, lam):
        lam_e = lam.unsqueeze(-1)
        gate = self.gate(torch.cat([z_t, z_f, lam_e], dim=-1))
        z_cross = gate * self.proj_t(z_t) + (1 - gate) * self.proj_f(z_f)
        z_mean = 0.5 * (z_t + z_f)
        z_fused = (1.0 - lam_e) * z_cross + lam_e * z_mean
        return self.out_proj(z_fused)


# ──────────────────────────────────────────────────────────────────────────────
# Full Model
# ──────────────────────────────────────────────────────────────────────────────
class ReconcileHAR(nn.Module):
    def __init__(
        self,
        n_channels,
        n_classes,
        stft_bins_per_scale,
        stft_kernel_sizes,
        stft_hop,
        freq_cnn_dim,
        latent_n,
        latent_d,
        perceiver_heads,
        perceiver_blocks,
        perceiver_dim,
        dropout,
        lambda_floor,
        residual_branch_weight,
    ):
        super().__init__()
        self.residual_branch_weight = residual_branch_weight
        self.stft = MultiScaleLearnableSTFT(
            in_channels=n_channels, bins_per_scale=stft_bins_per_scale,
            kernel_sizes=stft_kernel_sizes, hop=stft_hop,
        )
        self.freq_encoder = FreqEncoder(out_dim=freq_cnn_dim)
        self.time_encoder = PerceiverStyleEncoder(
            input_dim=n_channels, latent_n=latent_n, latent_d=latent_d,
            n_heads=perceiver_heads, num_blocks=perceiver_blocks,
            output_dim=perceiver_dim, dropout=dropout,
        )
        assert freq_cnn_dim == perceiver_dim
        feat_dim = perceiver_dim
        self.fusion = ConflictAwareFusion(feat_dim=feat_dim, dropout=dropout)

        def build_classifier(drop_rate):
            return nn.Sequential(
                # nn.Linear(feat_dim, feat_dim), nn.GELU(),
                nn.Dropout(drop_rate),
                nn.Linear(feat_dim, n_classes),
            )

        self.cls_time = build_classifier(0.1)
        self.cls_freq = build_classifier(0.1)
        self.classifier = build_classifier(dropout)
        self.residual_proj = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.LayerNorm(feat_dim),
        )
        self.conflict_est = ConflictEstimator(
            n_features=5, lambda_floor=lambda_floor, n_classes=n_classes,
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.Conv1d, nn.Conv2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        spec = self.stft(x)
        z_f = self.freq_encoder(spec)

        z_t = self.time_encoder(x.permute(0, 2, 1))

        logits_t = self.cls_time(z_t)
        logits_f = self.cls_freq(z_f)
        p_t = F.softmax(logits_t, dim=-1)
        p_f = F.softmax(logits_f, dim=-1)

        lam = self.conflict_est(z_t, z_f, p_t, p_f)

        z_fused = self.fusion(z_t, z_f, lam)

        w = self.residual_branch_weight
        z_out = z_fused + w * self.residual_proj(z_t + z_f)
        logits = self.classifier(z_out)
        features = {
            'z_t': z_t, 'z_f': z_f, 'z_fused': z_fused,
            'lam': lam, 'p_t': p_t, 'p_f': p_f,
        }
        return logits, logits_t, logits_f, features


# ──────────────────────────────────────────────────────────────────────────────
# Loss
# ──────────────────────────────────────────────────────────────────────────────
class ReconcileLoss(nn.Module):
    def __init__(
        self,
        alpha_agree,
        lambda_floor,
        lambda_ent_weight,
        lambda_center_weight,
        lambda_center_target,
    ):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.alpha_agree = alpha_agree
        self.lambda_floor = lambda_floor
        self.lambda_ent_weight = lambda_ent_weight
        self.lambda_center_weight = lambda_center_weight
        self.lambda_center_target = lambda_center_target

    def forward(self, logits, logits_t, logits_f, features, labels):
        lam = features['lam']
        p_t = features['p_t']
        p_f = features['p_f']

        l_ce_main = self.ce(logits, labels)
        l_ce_t    = self.ce(logits_t, labels)
        l_ce_f    = self.ce(logits_f, labels)
        l_ce = l_ce_main + 0.3 * (l_ce_t + l_ce_f)

        kl_tf = F.kl_div((p_t + 1e-8).log(), p_f, reduction='none').sum(dim=-1)
        kl_ft = F.kl_div((p_f + 1e-8).log(), p_t, reduction='none').sum(dim=-1)
        l_agree_raw = 0.5 * (kl_tf + kl_ft)
        l_agree = ((1.0 - lam) * l_agree_raw).mean()

        floor = self.lambda_floor
        p = ((lam - floor) / (1.0 - floor + 1e-8)).clamp(1e-6, 1 - 1e-6)
        ent = -(p * p.log() + (1 - p) * (1 - p).log()).mean()
        l_lam_ent = -self.lambda_ent_weight * ent

        l_lam_center = self.lambda_center_weight * (
            lam.mean() - self.lambda_center_target
        ).pow(2)

        total = l_ce + self.alpha_agree * l_agree + l_lam_ent + l_lam_center

        losses = {
            'ce_main': l_ce_main,
            'ce_t': l_ce_t,
            'ce_f': l_ce_f,
            'agree': l_agree,
            'lam_ent': l_lam_ent,
            'lam_center': l_lam_center,
            'total': total
        }

        return total, losses


# ──────────────────────────────────────────────────────────────────────────────
# Scheduler
# ──────────────────────────────────────────────────────────────────────────────
class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.optimizer  = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = optimizer.param_groups[0]['lr']
        self.min_lr = min_lr

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            lr = self.base_lr * (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / max(1, self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        return lr


# ──────────────────────────────────────────────────────────────────────────────
# Training & Evaluation
# ──────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_accum = defaultdict(float)
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits, logits_t, logits_f, features = model(x)
        loss, losses = criterion(logits, logits_t, logits_f, features, y)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        B = x.size(0)
        for k, v in losses.items():
            loss_accum[k] += v.item() * B
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    n = len(all_labels)
    avg_losses = {k: v / n for k, v in loss_accum.items()}
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    return avg_losses, acc, f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_accum = defaultdict(float)
    all_preds, all_labels, all_lam = [], [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits, logits_t, logits_f, features = model(x)
        loss, losses = criterion(logits, logits_t, logits_f, features, y)

        B = x.size(0)
        for k, v in losses.items():
            loss_accum[k] += v.item() * B
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_lam.extend(features['lam'].cpu().numpy())

    n = len(all_labels)
    avg_losses = {k: v / n for k, v in loss_accum.items()}
    metrics = {
        'acc': accuracy_score(all_labels, all_preds),
        'f1': f1_score(all_labels, all_preds, average='macro'),
        'precision': precision_score(all_labels, all_preds, average='macro'),
        'recall': recall_score(all_labels, all_preds, average='macro'),
        'mean_lambda': np.mean(all_lam),
        'std_lambda': np.std(all_lam),
    }
    return avg_losses, metrics, np.array(all_preds), np.array(all_labels)


@torch.no_grad()
def evaluate_branches(model, loader, device):
    model.eval()
    all_t, all_f, all_y = [], [], []
    for x, y in loader:
        x = x.to(device)
        _, logits_t, logits_f, _ = model(x)
        all_t.extend(logits_t.argmax(dim=-1).cpu().numpy())
        all_f.extend(logits_f.argmax(dim=-1).cpu().numpy())
        all_y.extend(y.numpy())
    y = np.array(all_y)
    return {
        'time_acc': accuracy_score(y, all_t),
        'freq_acc': accuracy_score(y, all_f),
        'time_f1': f1_score(y, all_t, average='macro'),
        'freq_f1': f1_score(y, all_f, average='macro'),
    }


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, epochs, patience):
    best_test_f1 = 0.0
    best_model_state = None
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        current_lr = scheduler.step(epoch - 1)

        _, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        _, test_metrics, _, _ = evaluate(
            model, test_loader, criterion, device
        )

        if epoch % 5 == 0 or epoch == 1:
            print(
                f"Ep {epoch:3d}/{epochs} | lr={current_lr:.4f} | "
                f"Tr acc={train_acc:.4f} F1={train_f1:.4f} | "
                f"Test acc={test_metrics['acc']:.4f} F1={test_metrics['f1']:.4f} | "
                f"λ={test_metrics['mean_lambda']:.3f}±{test_metrics['std_lambda']:.3f}"
            )

        if test_metrics["f1"] > best_test_f1:
            best_test_f1 = test_metrics["f1"]
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch} (best Test Macro F1={best_test_f1:.4f})")
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        model.to(device)

    return model, best_test_f1


def final_test(model, test_loader, criterion, device):
    _, test_metrics, _, _ = evaluate(model, test_loader, criterion, device)
    branch_metrics = evaluate_branches(model, test_loader, device)

    print(f"\nTest Macro F1: {test_metrics['f1']:.4f}")
    print(
        f"Test | Acc: {test_metrics['acc']:.4f} | "
        f"Macro F1: {test_metrics['f1']:.4f} | "
        f"Macro Precision: {test_metrics['precision']:.4f} | "
        f"Macro Recall: {test_metrics['recall']:.4f}"
    )
    print(
        f"Time Branch | Acc: {branch_metrics['time_acc']:.4f} | "
        f"Macro F1: {branch_metrics['time_f1']:.4f}"
    )
    print(
        f"Freq Branch | Acc: {branch_metrics['freq_acc']:.4f} | "
        f"Macro F1: {branch_metrics['freq_f1']:.4f}"
    )
    print(
        f"Lambda | Mean: {test_metrics['mean_lambda']:.4f} | "
        f"Std: {test_metrics['std_lambda']:.4f}"
    )

    return test_metrics, branch_metrics


# ──────────────────────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    data_dir = "/content/drive/MyDrive/Colab Notebooks/datasets/UCI-HAR"
    save_dir = "/content/drive/MyDrive/Colab Notebooks/HAR/Reconcile"
    ckpt_name = "reconcile_uci-har_best.pt"

    n_channels = 9
    n_classes = 6

    stft_bins_per_scale = 12
    stft_kernel_sizes = [16, 32]
    stft_hop = 8
    freq_cnn_dim = 48

    latent_n = 12
    latent_d = 48
    perceiver_heads = 4
    perceiver_blocks = 2
    perceiver_dim = 48
    dropout = 0.10

    batch_size = 64
    epochs = 100
    lr = 3e-4
    weight_decay = 2e-4
    warmup_epochs = 5

    alpha_agree = 0.5
    lambda_floor = 0.1
    lambda_ent_weight = 0.3
    lambda_center_weight = 0.3
    lambda_center_target = 0.5

    residual_branch_weight = 0.20

    seed = 42
    num_workers = 2 if USE_GPU else 0
    pin_memory = USE_GPU
    patience = 20

    set_seed(seed)

    print(f"\n{'='*70}")
    print("Reconcile-HAR Basecode")
    print(f"Device: {DEVICE}")
    print(f"{'='*70}\n")

    train_dataset, test_dataset = load_uci_har_dataset(data_dir)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=True
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    model = ReconcileHAR(
        n_channels=n_channels,
        n_classes=n_classes,
        stft_bins_per_scale=stft_bins_per_scale,
        stft_kernel_sizes=stft_kernel_sizes,
        stft_hop=stft_hop,
        freq_cnn_dim=freq_cnn_dim,
        latent_n=latent_n,
        latent_d=latent_d,
        perceiver_heads=perceiver_heads,
        perceiver_blocks=perceiver_blocks,
        perceiver_dim=perceiver_dim,
        dropout=dropout,
        lambda_floor=lambda_floor,
        residual_branch_weight=residual_branch_weight,
    ).to(DEVICE)

    criterion = ReconcileLoss(
        alpha_agree=alpha_agree,
        lambda_floor=lambda_floor,
        lambda_ent_weight=lambda_ent_weight,
        lambda_center_weight=lambda_center_weight,
        lambda_center_target=lambda_center_target,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    scheduler = CosineWarmupScheduler(
        optimizer=optimizer,
        warmup_epochs=warmup_epochs,
        total_epochs=epochs
    )

    print(f"Model params: {count_parameters(model):,}\n")

    model, best_test_f1 = train_model(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        epochs=epochs,
        patience=patience
    )

    print(f"\nBest Test Macro F1: {best_test_f1:.4f}")
    final_test(model, test_loader, criterion, DEVICE)

    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(save_dir, ckpt_name))

    print(f"\nSaved to: {os.path.join(save_dir, ckpt_name)}")

Device: cuda | pin_memory: True

Reconcile-HAR Basecode
Device: cuda

Train dataset size: 7352, Test dataset size: 2947
Model params: 105,251

Ep   1/100 | lr=0.0001 | Tr acc=0.6439 F1=0.5719 | Test acc=0.7801 F1=0.7560 | λ=0.547±0.000
Ep   5/100 | lr=0.0003 | Tr acc=0.9459 F1=0.9492 | Test acc=0.8914 F1=0.8912 | λ=0.562±0.000
Ep  10/100 | lr=0.0003 | Tr acc=0.9590 F1=0.9620 | Test acc=0.9128 F1=0.9124 | λ=0.549±0.011
Ep  15/100 | lr=0.0003 | Tr acc=0.9703 F1=0.9726 | Test acc=0.9308 F1=0.9302 | λ=0.552±0.034
Ep  20/100 | lr=0.0003 | Tr acc=0.9789 F1=0.9805 | Test acc=0.9525 F1=0.9524 | λ=0.558±0.052
Ep  25/100 | lr=0.0003 | Tr acc=0.9819 F1=0.9833 | Test acc=0.9623 F1=0.9629 | λ=0.566±0.069
Ep  30/100 | lr=0.0003 | Tr acc=0.9892 F1=0.9900 | Test acc=0.9623 F1=0.9623 | λ=0.567±0.078
Ep  35/100 | lr=0.0002 | Tr acc=0.9899 F1=0.9907 | Test acc=0.9569 F1=0.9572 | λ=0.571±0.091
Ep  40/100 | lr=0.0002 | Tr acc=0.9947 F1=0.9951 | Test acc=0.9498 F1=0.9504 | λ=0.576±0.105

Early stopping at e